In [11]:
import pandas as pd
import glob
import os
import re
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Load data
folder_path = 'raw_data'
all_files = glob.glob(os.path.join(folder_path, "asrs_*.csv"))
print(f"Found {len(all_files)} files. Joining...")
df = pd.concat([pd.read_csv(file, low_memory=False) for file in all_files], axis=0, ignore_index=True)

# Clean column names (to snake_case)
df.columns = (
    df.columns.str.strip()
    .str.replace(r'[^a-zA-Z0-9\s/.]', '', regex=True)
    .str.replace(r' \/ ', '_', regex=True)
    .str.replace(r'\.', '_', regex=True)
    .str.replace(r'\s+', '_', regex=True)
    .str.lower()
)

Found 2 files. Joining...


In [12]:
# 1. Drop columns with >90% nulls
df = df.dropna(axis=1, thresh=len(df) * 0.10)

# 2. Drop specific unused columns right away to save processing time
cols_to_drop = [
    'asrs_report_number_accession_number', 'callback', 'callback_1', 'experience',
    'qualification', 'qualification_1', 'miss_distance', 'route_in_use',
    'aircraft_reference', 'location_of_person', 'location_of_person_1', 'weather_elements_visibility'
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# 3. Merge Narratives
if 'narrative_1' in df.columns:
    df['full_narrative'] = df['narrative'].fillna('') + " | " + df['narrative_1'].fillna('')
    df = df.drop(columns=['narrative', 'narrative_1'])
elif 'narrative' in df.columns:
    df['full_narrative'] = df['narrative']
    df = df.drop(columns=['narrative'])
df['full_narrative'] = df['full_narrative'].str.strip(' |')

# 4. Handle Dates & Fill Numerical NaNs with Median
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'].astype(str), format='%Y%m', errors='coerce')

num_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# 5. Global String Formatting (Lowercase & Nulls)
string_cols = df.select_dtypes(include=['object', 'string']).columns
df[string_cols] = df[string_cols].apply(lambda x: x.str.lower())
df = df.replace(['unknown', 0, 0.0], np.nan) # Handles 'unknown' text and 0 AGL altitudes globally

In [13]:
# 1. Clean 'ZZZ' redactions and literal header artifacts
scrub_cols = ['atc_advisory', 'atc_advisory_1', 'airspace', 'airspace_1', 'locale_reference']
for col in scrub_cols:
    if col in df.columns:
        df[col] = df[col].replace(col, np.nan) # Remove stray header names
        if col == 'locale_reference':
            df[col] = df[col].str.split('.').str[-1] # Remove ZZZ. prefix
        df[col] = df[col].str.replace(r'\bZ+\d*\b', '', regex=True).str.strip() # Remove standalone ZZZ
        df[col] = df[col].replace('', np.nan)

# 2. Standardize Airspace (Extract 'class x')
for col in ['airspace', 'airspace_1']:
    if col in df.columns:
        df[col] = df[col].str.replace(r'(class\s+[a-z])\s+.*', r'\1', regex=True).str.strip()

# 3. Keep Only the Primary Item (First string before the semicolon)
primary_val_cols = ['problem', 'flight_phase', 'flight_phase_1', 'detector', 'when_detected', 'atc_advisory', 'atc_advisory_1']
for col in primary_val_cols:
    if col in df.columns:
        df[col] = df[col].replace(col, np.nan) # Catch any remaining header artifacts
        df[col] = df[col].str.split(';').str[0].str.strip()
        if 'atc_advisory' in col:
            df[col] = df[col].str.split(' ').str[0] # Additional split for ATC

In [14]:
# 1. Expand list columns into boolean (0/1) columns
list_cols = [
    'anomaly', 'function', 'human_factors', 'communication_breakdown',
    'function_1', 'human_factors_1', 'result',
    'contributing_factors_situations'
]

for col in list_cols:
    if col in df.columns:
        dummies = (
            df[col].fillna('')
            .str.replace(r'\s*;\s*', ';', regex=True)
            .str.get_dummies(sep=';')
        )
        # Rename new columns: original_col_name + sanitized_value
        dummies.columns = [f"{col}_" + re.sub(r'[^a-z0-9]+', '_', val).strip('_') for val in dummies.columns]
        df = pd.concat([df, dummies], axis=1).drop(columns=[col])


In [15]:
# Find all duplicate column names
duplicate_cols = df.columns[df.columns.duplicated()].tolist()

if duplicate_cols:
    print(f"Found {len(duplicate_cols)} duplicate column(s).")
    # Using set() to show the unique names of the duplicates without printing it 50 times
    print("Duplicate names:", set(duplicate_cols))

    # 2. Keep the first occurrence of each column and drop the rest
    df = df.loc[:, ~df.columns.duplicated(keep='first')]

    print(f"\nDuplicates removed. Final DataFrame shape: {df.shape}")
else:
    print("No duplicate columns found. DataFrame shape:", df.shape)

No duplicate columns found. DataFrame shape: (5883, 304)


In [16]:
print(f"Final DataFrame shape: {df.shape}")

# 2. Export to CSV
os.makedirs("clean_data", exist_ok=True)
df.to_csv("clean_data/asrs_2025_cleaned.csv", index=False, na_rep="")
print("Export complete!")

Final DataFrame shape: (5883, 304)
Export complete!


In [18]:
my_list = df.columns.tolist()
print("[", end="") # Start the bracket
for i, item in enumerate(my_list):
    # Print the item with quotes
    # Using repr() ensures strings get their single quotes ''
    print(repr(item), end="")

    # Add a comma if it's not the last item
    if i < len(my_list) - 1:
        print(", ", end="")

    # Every 10 items (and not at the very end), add a newline
    if (i + 1) % 10 == 0 and i < len(my_list) - 1:
        print("\n ", end="") # Space after \n to align with the start [

print("]") # End the bracket

['acn', 'date', 'local_time_of_day', 'locale_reference', 'state_reference', 'relative_position_distance_nautical_miles', 'altitude_agl_single_value', 'altitude_msl_single_value', 'flight_conditions', 'light', 
 'ceiling', 'atc_advisory', 'aircraft_operator', 'make_model_name', 'crew_size', 'operating_under_far_part', 'flight_plan', 'mission', 'flight_phase', 'airspace', 
 'aircraft_component', 'problem', 'atc_advisory_1', 'aircraft_operator_1', 'make_model_name_1', 'crew_size_1', 'operating_under_far_part_1', 'flight_phase_1', 'airspace_1', 'location_in_aircraft', 
 'reporter_organization', 'location_in_aircraft_1', 'reporter_organization_1', 'asrs_report_number_accession_number_1', 'detector', 'when_detected', 'primary_problem', 'synopsis', 'full_narrative', 'anomaly_aircraft_equipment_problem_critical', 
 'anomaly_aircraft_equipment_problem_less_severe', 'anomaly_airspace_violation_all_types', 'anomaly_atc_issue_all_types', 'anomaly_conflict_airborne_conflict', 'anomaly_conflict_grou